In [3]:
# pip install lightgbm scikit-learn pandas numpy

import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =========================
# 1. Load data
# =========================

DATA_PATH = "DataSet.csv"
TARGET = "F3924"

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
print("Target distribution:")
print(df[TARGET].value_counts())

# =========================
# 2. Basic cleaning
# =========================

drop_cols = []

if "Unnamed: 0" in df.columns:
    drop_cols.append("Unnamed: 0")

# possible leakage column
if "F2230" in df.columns:
    drop_cols.append("F2230")

empty_cols = df.columns[df.isna().mean() == 1].tolist()
drop_cols += empty_cols

df = df.drop(columns=list(set(drop_cols)), errors="ignore")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# =========================
# 3. Feature engineering
# =========================

if "F3888" in X.columns:
    date_col = pd.to_datetime(X["F3888"], errors="coerce")
    max_date = date_col.max()
    X["F3888_days_old"] = (max_date - date_col).dt.days
    X = X.drop(columns=["F3888"])

if "F3889" in X.columns:
    X["F3889_days"] = X["F3889"].astype(str).str.extract(r"(\d+)").astype(float)
    X["F3889_type"] = X["F3889"].astype(str).str[0]

missing_rate = X.isna().mean()

high_missing_cols = missing_rate[missing_rate > 0.95].index.tolist()
X = X.drop(columns=high_missing_cols)

missing_rate = X.isna().mean()
missing_cols = missing_rate[(missing_rate > 0.05) & (missing_rate < 0.95)].index.tolist()

if len(missing_cols) > 0:
    missing_indicators = X[missing_cols].isna().astype("int8")
    missing_indicators.columns = [col + "_missing" for col in missing_cols]
    X = pd.concat([X, missing_indicators], axis=1)

X = X.copy()

constant_cols = X.columns[X.nunique(dropna=False) <= 1].tolist()
X = X.drop(columns=constant_cols)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype("category")

print("Final shape:", X.shape)
print("Categorical columns:", cat_cols)

# =========================
# 4. LightGBM training
# =========================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
models = []

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 5,
    "lambda_l1": 1.0,
    "lambda_l2": 5.0,
    "scale_pos_weight": scale_pos_weight,
    "verbose": -1,
    "seed": 42
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    model = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        valid_names=["valid"],
        num_boost_round=3000,
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(100)
        ]
    )

    pred = model.predict(X_val, num_iteration=model.best_iteration)
    oof_pred[val_idx] = pred
    models.append(model)

    print("Fold ROC-AUC:", roc_auc_score(y_val, pred))
    print("Fold PR-AUC:", average_precision_score(y_val, pred))

# =========================
# 5. Overall evaluation
# =========================

print("\n========== Overall Metrics ==========")

overall_roc_auc = roc_auc_score(y, oof_pred)
overall_pr_auc = average_precision_score(y, oof_pred)

print("Overall ROC-AUC:", overall_roc_auc)
print("Overall PR-AUC:", overall_pr_auc)

# =========================
# 6. Best threshold search
# =========================

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.01, 0.99, 0.01):
    pred_label = (oof_pred >= t).astype(int)
    score = f1_score(y, pred_label)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("\nBest Threshold:", best_threshold)
print("Best F1:", best_f1)

final_pred = (oof_pred >= best_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(y, final_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y, final_pred))

# =========================
# 7. Risk score output
# =========================

risk_output = pd.DataFrame({
    "actual_label": y.values,
    "risk_probability": oof_pred,
    "risk_score_0_100": (oof_pred * 100).round(2),
    "predicted_label": final_pred
})

risk_output.to_csv("mule_account_risk_scores.csv", index=False)

print("\nSaved file: mule_account_risk_scores.csv")

# =========================
# 8. Feature importance
# =========================

feature_importance_values = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0
)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance_values
}).sort_values(by="importance", ascending=False)

importance.to_csv("feature_importance.csv", index=False)

print("Saved file: feature_importance.csv")

print("\nTop 20 important features:")
print(importance.head(20))

Original shape: (9082, 3925)
Target distribution:
F3924
0    9001
1      81
Name: count, dtype: int64
Final shape: (9082, 4696)
Categorical columns: ['F3886', 'F3889', 'F3890', 'F3891', 'F3892', 'F3893', 'F3889_type']
scale_pos_weight: 111.12345679012346

========== Fold 1 ==========
Training until validation scores don't improve for 150 rounds
[100]	valid's auc: 0.999063	valid's binary_logloss: 0.010711
[200]	valid's auc: 0.999965	valid's binary_logloss: 0.00447548
[300]	valid's auc: 1	valid's binary_logloss: 0.00243095
Early stopping, best iteration is:
[235]	valid's auc: 1	valid's binary_logloss: 0.00352061
Fold ROC-AUC: 1.0
Fold PR-AUC: 1.0

========== Fold 2 ==========
Training until validation scores don't improve for 150 rounds
[100]	valid's auc: 0.995588	valid's binary_logloss: 0.0108784
[200]	valid's auc: 0.998235	valid's binary_logloss: 0.00756586
[300]	valid's auc: 0.998497	valid's binary_logloss: 0.00697781
[400]	valid's auc: 0.999673	valid's binary_logloss: 0.00680545
[500

In [4]:
#leakage-check code:f top features show crazy individual AUC, we may need to remove them and retrain.

results = []

for col in X.columns:
    try:
        feature = X[col]

        # Convert categories to numbers
        if str(feature.dtype) == "category":
            feature = feature.cat.codes

        feature = pd.to_numeric(feature, errors="coerce")

        if feature.nunique(dropna=True) <= 1:
            continue

        feature = feature.fillna(feature.median())

        auc = roc_auc_score(y, feature)

        # Handle reversed direction
        auc = max(auc, 1 - auc)

        pr_auc = average_precision_score(y, feature)

        results.append([
            col,
            auc,
            pr_auc
        ])

    except:
        pass

results = pd.DataFrame(
    results,
    columns=[
        "feature",
        "single_feature_auc",
        "single_feature_pr_auc"
    ]
)

results = results.sort_values(
    "single_feature_auc",
    ascending=False
)

print(results.head(30))

results.to_csv(
    "single_feature_leakage_check.csv",
    index=False
)

print("Saved: single_feature_leakage_check.csv")

     feature  single_feature_auc  single_feature_pr_auc
3267   F3912            0.987488               0.939847
3170   F3811            0.827293               0.004993
3164   F3805            0.824261               0.005010
3158   F3799            0.817884               0.005038
1479   F1813            0.816293               0.005058
1577   F1921            0.812645               0.005108
1381   F1705            0.811576               0.005083
783    F1057            0.810674               0.005200
683     F949            0.809998               0.005179
1675   F2029            0.809793               0.005119
883    F1165            0.808666               0.005168
1283   F1597            0.808170               0.005100
3165   F3806            0.804272               0.005130
3159   F3800            0.804158               0.005121
1480   F1814            0.804081               0.005129
3160   F3801            0.801817               0.005147
1773   F2137            0.801763               0

In [5]:
#I wanted to see if there was a feature that alone predicts the target almost perfectly. it looks like 0.999 or 1 which solely one feature can predict the output
#F3912 = 0.987

#This is very strong.

#But not necessarily leakage.# pip install lightgbm scikit-learn pandas numpy

import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix
)

DATA_PATH = "DataSet.csv"
TARGET = "F3924"

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
print("Target distribution:")
print(df[TARGET].value_counts())

# =========================
# Cleaning
# =========================

drop_cols = []

if "Unnamed: 0" in df.columns:
    drop_cols.append("Unnamed: 0")

if "F2230" in df.columns:
    drop_cols.append("F2230")

# Drop F3912 to test possible leakage
if "F3912" in df.columns:
    drop_cols.append("F3912")

empty_cols = df.columns[df.isna().mean() == 1].tolist()
drop_cols += empty_cols

df = df.drop(columns=list(set(drop_cols)), errors="ignore")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# =========================
# Feature Engineering
# =========================

if "F3888" in X.columns:
    date_col = pd.to_datetime(X["F3888"], errors="coerce")
    max_date = date_col.max()
    X["F3888_days_old"] = (max_date - date_col).dt.days
    X = X.drop(columns=["F3888"])

if "F3889" in X.columns:
    X["F3889_days"] = X["F3889"].astype(str).str.extract(r"(\d+)").astype(float)
    X["F3889_type"] = X["F3889"].astype(str).str[0]

missing_rate = X.isna().mean()

high_missing_cols = missing_rate[missing_rate > 0.95].index.tolist()
X = X.drop(columns=high_missing_cols)

missing_rate = X.isna().mean()
missing_cols = missing_rate[(missing_rate > 0.05) & (missing_rate < 0.95)].index.tolist()

if len(missing_cols) > 0:
    missing_indicators = X[missing_cols].isna().astype("int8")
    missing_indicators.columns = [col + "_missing" for col in missing_cols]
    X = pd.concat([X, missing_indicators], axis=1)

X = X.copy()

constant_cols = X.columns[X.nunique(dropna=False) <= 1].tolist()
X = X.drop(columns=constant_cols)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype("category")

print("Final shape:", X.shape)
print("Categorical columns:", cat_cols)
print("Dropped columns:", drop_cols)

# =========================
# Model Training
# =========================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_pred = np.zeros(len(X))
models = []

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 5,
    "lambda_l1": 1.0,
    "lambda_l2": 5.0,
    "scale_pos_weight": scale_pos_weight,
    "verbose": -1,
    "seed": 42
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    model = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        valid_names=["valid"],
        num_boost_round=3000,
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(100)
        ]
    )

    pred = model.predict(X_val, num_iteration=model.best_iteration)
    oof_pred[val_idx] = pred
    models.append(model)

    print("Fold ROC-AUC:", roc_auc_score(y_val, pred))
    print("Fold PR-AUC:", average_precision_score(y_val, pred))

# =========================
# Evaluation
# =========================

print("\n========== WITHOUT F3912 RESULTS ==========")

overall_roc_auc = roc_auc_score(y, oof_pred)
overall_pr_auc = average_precision_score(y, oof_pred)

print("Overall ROC-AUC:", overall_roc_auc)
print("Overall PR-AUC:", overall_pr_auc)

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.01, 0.99, 0.01):
    pred_label = (oof_pred >= t).astype(int)
    score = f1_score(y, pred_label)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("\nBest Threshold:", best_threshold)
print("Best F1:", best_f1)

final_pred = (oof_pred >= best_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(y, final_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y, final_pred))

# =========================
# Output Files
# =========================

risk_output = pd.DataFrame({
    "actual_label": y.values,
    "risk_probability": oof_pred,
    "risk_score_0_100": (oof_pred * 100).round(2),
    "predicted_label": final_pred
})

risk_output.to_csv("mule_account_risk_scores_without_F3912.csv", index=False)

feature_importance_values = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0
)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance_values
}).sort_values(by="importance", ascending=False)

importance.to_csv("feature_importance_without_F3912.csv", index=False)

print("\nSaved: mule_account_risk_scores_without_F3912.csv")
print("Saved: feature_importance_without_F3912.csv")

print("\nTop 20 important features:")
print(importance.head(20))

Original shape: (9082, 3925)
Target distribution:
F3924
0    9001
1      81
Name: count, dtype: int64
Final shape: (9082, 4695)
Categorical columns: ['F3886', 'F3889', 'F3890', 'F3891', 'F3892', 'F3893', 'F3889_type']
Dropped columns: ['Unnamed: 0', 'F2230', 'F3912', 'F128', 'F131', 'F182', 'F185', 'F189', 'F192', 'F236', 'F239', 'F290', 'F293', 'F390', 'F393', 'F437', 'F440', 'F492', 'F495', 'F539', 'F542', 'F594', 'F597', 'F2312', 'F2360', 'F2455', 'F2458', 'F2552', 'F2555', 'F2607', 'F2655', 'F2707', 'F2753', 'F2756', 'F2807', 'F2810', 'F2814', 'F2860', 'F2863', 'F2914', 'F2917', 'F2921', 'F2924', 'F2968', 'F2971', 'F3027', 'F3030', 'F3074', 'F3077', 'F3133', 'F3179', 'F3182', 'F3233', 'F3236', 'F3341', 'F3344', 'F3449', 'F3452', 'F3503', 'F3506', 'F3557', 'F3560', 'F3665', 'F3668', 'F3773', 'F3776']
scale_pos_weight: 111.12345679012346

========== Fold 1 ==========
Training until validation scores don't improve for 150 rounds
[100]	valid's auc: 0.931323	valid's binary_logloss: 0.03

In [6]:
#Recall
#98.8%  →  67.9%
#it means : F3912 is extremely informative and there is no data leakage

In [1]:
# pip install lightgbm catboost scikit-learn pandas numpy

import pandas as pd
import numpy as np
import lightgbm as lgb

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve
)

# =========================
# 1. Load Data
# =========================

DATA_PATH = "DataSet.csv"
TARGET = "F3924"

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
print("Target distribution:")
print(df[TARGET].value_counts())

# =========================
# 2. Cleaning
# =========================

drop_cols = []

if "Unnamed: 0" in df.columns:
    drop_cols.append("Unnamed: 0")

# possible leakage / ID-like suspicious column
if "F2230" in df.columns:
    drop_cols.append("F2230")

empty_cols = df.columns[df.isna().mean() == 1].tolist()
drop_cols += empty_cols

df = df.drop(columns=list(set(drop_cols)), errors="ignore")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# =========================
# 3. Feature Engineering
# =========================

if "F3888" in X.columns:
    date_col = pd.to_datetime(X["F3888"], errors="coerce")
    max_date = date_col.max()
    X["F3888_days_old"] = (max_date - date_col).dt.days
    X = X.drop(columns=["F3888"])

if "F3889" in X.columns:
    X["F3889_days"] = X["F3889"].astype(str).str.extract(r"(\d+)").astype(float)
    X["F3889_type"] = X["F3889"].astype(str).str[0]

# Drop very high missing columns
missing_rate = X.isna().mean()
high_missing_cols = missing_rate[missing_rate > 0.95].index.tolist()
X = X.drop(columns=high_missing_cols)

# Missing indicators
missing_rate = X.isna().mean()
missing_cols = missing_rate[(missing_rate > 0.05) & (missing_rate < 0.95)].index.tolist()

if len(missing_cols) > 0:
    missing_indicators = X[missing_cols].isna().astype("int8")
    missing_indicators.columns = [col + "_missing" for col in missing_cols]
    X = pd.concat([X, missing_indicators], axis=1)

X = X.copy()

# Drop constant columns
constant_cols = X.columns[X.nunique(dropna=False) <= 1].tolist()
X = X.drop(columns=constant_cols)

# Detect categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype("category")

cat_feature_indices = [X.columns.get_loc(col) for col in cat_cols]

print("Final shape:", X.shape)
print("Categorical columns:", cat_cols)

# =========================
# 4. CV Setup
# =========================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

lgb_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))
ensemble_oof = np.zeros(len(X))

lgb_models = []
cat_models = []

# =========================
# 5. LightGBM Params
# =========================

lgb_params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",
    "learning_rate": 0.015,
    "num_leaves": 24,
    "max_depth": -1,
    "min_data_in_leaf": 15,
    "feature_fraction": 0.80,
    "bagging_fraction": 0.80,
    "bagging_freq": 5,
    "lambda_l1": 1.5,
    "lambda_l2": 6.0,
    "scale_pos_weight": scale_pos_weight,
    "verbose": -1,
    "seed": 42
}

# =========================
# 6. Train LightGBM + CatBoost
# =========================

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # ---------- LightGBM ----------
    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    lgb_model = lgb.train(
        lgb_params,
        train_data,
        valid_sets=[val_data],
        valid_names=["valid"],
        num_boost_round=4000,
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200)
        ]
    )

    lgb_pred = lgb_model.predict(
        X_val,
        num_iteration=lgb_model.best_iteration
    )

    lgb_oof[val_idx] = lgb_pred
    lgb_models.append(lgb_model)

    print("LightGBM ROC-AUC:", roc_auc_score(y_val, lgb_pred))
    print("LightGBM PR-AUC :", average_precision_score(y_val, lgb_pred))

    # ---------- CatBoost ----------
    X_train_cat = X_train.copy()
    X_val_cat = X_val.copy()

    for col in cat_cols:
        X_train_cat[col] = X_train_cat[col].astype(str).fillna("missing")
        X_val_cat[col] = X_val_cat[col].astype(str).fillna("missing")

    cat_model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.02,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=[1, scale_pos_weight],
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        l2_leaf_reg=8
    )

    cat_model.fit(
        X_train_cat,
        y_train,
        eval_set=(X_val_cat, y_val),
        cat_features=cat_feature_indices,
        use_best_model=True
    )

    cat_pred = cat_model.predict_proba(X_val_cat)[:, 1]

    cat_oof[val_idx] = cat_pred
    cat_models.append(cat_model)

    print("CatBoost ROC-AUC:", roc_auc_score(y_val, cat_pred))
    print("CatBoost PR-AUC :", average_precision_score(y_val, cat_pred))

# =========================
# 7. Ensemble
# =========================

# Try multiple weights and pick best PR-AUC
best_weight = 0.5
best_pr_auc = 0

for w in np.arange(0.0, 1.01, 0.05):
    temp_pred = (w * lgb_oof) + ((1 - w) * cat_oof)
    score = average_precision_score(y, temp_pred)

    if score > best_pr_auc:
        best_pr_auc = score
        best_weight = w

ensemble_oof = (best_weight * lgb_oof) + ((1 - best_weight) * cat_oof)

print("\n========== MODEL COMPARISON ==========")
print("LightGBM ROC-AUC:", roc_auc_score(y, lgb_oof))
print("LightGBM PR-AUC :", average_precision_score(y, lgb_oof))

print("CatBoost ROC-AUC:", roc_auc_score(y, cat_oof))
print("CatBoost PR-AUC :", average_precision_score(y, cat_oof))

print("Best Ensemble Weight for LightGBM:", best_weight)
print("Ensemble ROC-AUC:", roc_auc_score(y, ensemble_oof))
print("Ensemble PR-AUC :", average_precision_score(y, ensemble_oof))

# =========================
# 8. Threshold Optimization
# =========================

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.01, 0.99, 0.01):
    pred_label = (ensemble_oof >= t).astype(int)
    score = f1_score(y, pred_label)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("\nBest Threshold:", best_threshold)
print("Best F1:", best_f1)

final_pred = (ensemble_oof >= best_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(y, final_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y, final_pred))

# =========================
# 9. Risk Score Output
# =========================

risk_output = pd.DataFrame({
    "actual_label": y.values,
    "lgb_probability": lgb_oof,
    "cat_probability": cat_oof,
    "ensemble_probability": ensemble_oof,
    "risk_score_0_100": (ensemble_oof * 100).round(2),
    "predicted_label": final_pred
})

risk_output.to_csv("optimized_mule_account_risk_scores.csv", index=False)

print("\nSaved: optimized_mule_account_risk_scores.csv")

# =========================
# 10. Feature Importance
# =========================

lgb_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": np.mean(
        [model.feature_importance(importance_type="gain") for model in lgb_models],
        axis=0
    )
}).sort_values("importance", ascending=False)

lgb_importance.to_csv("optimized_lgb_feature_importance.csv", index=False)

print("Saved: optimized_lgb_feature_importance.csv")

print("\nTop 20 LightGBM features:")
print(lgb_importance.head(20))

Original shape: (9082, 3925)
Target distribution:
F3924
0    9001
1      81
Name: count, dtype: int64
Final shape: (9082, 4696)
Categorical columns: ['F3886', 'F3889', 'F3890', 'F3891', 'F3892', 'F3893', 'F3889_type']
scale_pos_weight: 111.12345679012346

========== Fold 1 ==========
Training until validation scores don't improve for 200 rounds
[200]	valid's auc: 1	valid's binary_logloss: 0.00730619
Early stopping, best iteration is:
[1]	valid's auc: 1	valid's binary_logloss: 0.0374427
LightGBM ROC-AUC: 1.0
LightGBM PR-AUC : 1.0
0:	test: 0.9999479	best: 0.9999479 (0)	total: 765ms	remaining: 38m 14s
200:	test: 1.0000000	best: 1.0000000 (1)	total: 1m 41s	remaining: 23m 29s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1
bestIteration = 1

Shrink model to first 2 iterations.
CatBoost ROC-AUC: 1.0
CatBoost PR-AUC : 1.0

========== Fold 2 ==========
Training until validation scores don't improve for 200 rounds
[200]	valid's auc: 0.998758	valid's binary_logloss: 0.008090

In [4]:

# pip install lightgbm scikit-learn pandas numpy

import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

DATA_PATH = "DataSet.csv"
TARGET = "F3924"

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
print("Target distribution:")
print(df[TARGET].value_counts())

# =========================
# Cleaning
# =========================

drop_cols = []

if "Unnamed: 0" in df.columns:
    drop_cols.append("Unnamed: 0")

if "F2230" in df.columns:
    drop_cols.append("F2230")

empty_cols = df.columns[df.isna().mean() == 1].tolist()
drop_cols += empty_cols

df = df.drop(columns=list(set(drop_cols)), errors="ignore")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# =========================
# Feature Engineering
# =========================

if "F3888" in X.columns:
    date_col = pd.to_datetime(X["F3888"], errors="coerce")
    max_date = date_col.max()
    X["F3888_days_old"] = (max_date - date_col).dt.days
    X = X.drop(columns=["F3888"])

if "F3889" in X.columns:
    X["F3889_days"] = X["F3889"].astype(str).str.extract(r"(\d+)").astype(float)
    X["F3889_type"] = X["F3889"].astype(str).str[0]

missing_rate = X.isna().mean()

high_missing_cols = missing_rate[missing_rate > 0.95].index.tolist()
X = X.drop(columns=high_missing_cols)

missing_rate = X.isna().mean()
missing_cols = missing_rate[(missing_rate > 0.05) & (missing_rate < 0.95)].index.tolist()

if len(missing_cols) > 0:
    missing_indicators = X[missing_cols].isna().astype("int8")
    missing_indicators.columns = [col + "_missing" for col in missing_cols]
    X = pd.concat([X, missing_indicators], axis=1)

X = X.copy()

constant_cols = X.columns[X.nunique(dropna=False) <= 1].tolist()
X = X.drop(columns=constant_cols)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype("category")

print("Final shape:", X.shape)
print("Categorical columns:", cat_cols)

# =========================
# Model Training
# =========================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_pred = np.zeros(len(X))
models = []

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 5,
    "lambda_l1": 1.0,
    "lambda_l2": 5.0,
    "scale_pos_weight": scale_pos_weight,
    "verbose": -1,
    "seed": 42
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    model = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        valid_names=["valid"],
        num_boost_round=3000,
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(100)
        ]
    )

    pred = model.predict(X_val, num_iteration=model.best_iteration)
    oof_pred[val_idx] = pred
    models.append(model)

    print("Fold ROC-AUC:", roc_auc_score(y_val, pred))
    print("Fold PR-AUC:", average_precision_score(y_val, pred))

# =========================
# Overall Metrics
# =========================

print("\n========== Overall Metrics ==========")

print("Overall ROC-AUC:", roc_auc_score(y, oof_pred))
print("Overall PR-AUC:", average_precision_score(y, oof_pred))

# =========================
# Best F1 Threshold
# =========================

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.01, 0.99, 0.01):
    pred_label = (oof_pred >= t).astype(int)
    score = f1_score(y, pred_label)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("\nBest F1 Threshold:", best_threshold)
print("Best F1:", best_f1)

best_pred = (oof_pred >= best_threshold).astype(int)

print("\nClassification Report at Best F1 Threshold:")
print(classification_report(y, best_pred))

print("\nConfusion Matrix at Best F1 Threshold:")
print(confusion_matrix(y, best_pred))

# =========================
# Threshold Test To Catch All 81 Mule Accounts
# =========================

print("\n========== Threshold Test ==========")

thresholds = [
    0.10, 0.09, 0.08, 0.07, 0.06,
    0.05, 0.04, 0.03, 0.02, 0.01,
    0.005, 0.001
]

best_recall_threshold = None

for t in thresholds:
    preds = (oof_pred >= t).astype(int)

    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()

    print("\n------------------------------")
    print("Threshold:", t)
    print("Confusion Matrix:")
    print(cm)
    print("Mule detected:", tp, "/ 81")
    print("Mule missed:", fn)
    print("False positives:", fp)
    print("Precision:", precision_score(y, preds, zero_division=0))
    print("Recall:", recall_score(y, preds, zero_division=0))
    print("F1:", f1_score(y, preds, zero_division=0))

    if fn == 0 and best_recall_threshold is None:
        best_recall_threshold = t

if best_recall_threshold is not None:
    final_threshold = best_recall_threshold
    print("\nFinal threshold chosen to catch all mule accounts:", final_threshold)
else:
    final_threshold = best_threshold
    print("\nNo tested threshold caught all mule accounts.")
    print("Using best F1 threshold:", final_threshold)

# =========================
# Final Prediction
# =========================

final_pred = (oof_pred >= final_threshold).astype(int)

print("\n========== Final Model Report ==========")
print("Final Threshold:", final_threshold)
print(classification_report(y, final_pred))
print("Confusion Matrix:")
print(confusion_matrix(y, final_pred))

# =========================
# Risk Score Output
# =========================

risk_output = pd.DataFrame({
    "actual_label": y.values,
    "risk_probability": oof_pred,
    "risk_score_0_100": (oof_pred * 100).round(2),
    "predicted_label": final_pred
})

risk_output.to_csv("final_mule_account_risk_scores.csv", index=False)

print("\nSaved: final_mule_account_risk_scores.csv")

# =========================
# Feature Importance
# =========================

feature_importance_values = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0
)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importance_values
}).sort_values(by="importance", ascending=False)

importance.to_csv("final_feature_importance.csv", index=False)

print("Saved: final_feature_importance.csv")

print("\nTop 20 important features:")
print(importance.head(20))


Original shape: (9082, 3925)
Target distribution:
F3924
0    9001
1      81
Name: count, dtype: int64
Final shape: (9082, 4696)
Categorical columns: ['F3886', 'F3889', 'F3890', 'F3891', 'F3892', 'F3893', 'F3889_type']
scale_pos_weight: 111.12345679012346

========== Fold 1 ==========
Training until validation scores don't improve for 150 rounds
[100]	valid's auc: 0.999063	valid's binary_logloss: 0.010711
[200]	valid's auc: 0.999965	valid's binary_logloss: 0.00447548
[300]	valid's auc: 1	valid's binary_logloss: 0.00243095
Early stopping, best iteration is:
[235]	valid's auc: 1	valid's binary_logloss: 0.00352061
Fold ROC-AUC: 1.0
Fold PR-AUC: 1.0

========== Fold 2 ==========
Training until validation scores don't improve for 150 rounds
[100]	valid's auc: 0.995588	valid's binary_logloss: 0.0108784
[200]	valid's auc: 0.998235	valid's binary_logloss: 0.00756586
[300]	valid's auc: 0.998497	valid's binary_logloss: 0.00697781
[400]	valid's auc: 0.999673	valid's binary_logloss: 0.00680545
[500